**Link al repositorio de GitHub:** [https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo](https://github.com/vgcarlol/Aprendizaje-Por-Refuerzo)

# CC3104 - Aprendizaje por Refuerzo
## Laboratorio 3 - Entrega Parcial

---
### Task 1: Diseño del Experimento

#### 1. Especificación del MDP
Para evaluar el uso de agentes de RL en el juego de rol táctico, modelaremos el entorno como un GridWorld de 6x6.
*   **Espacio de Estados $\mathcal{S}$:** Conjunto de coordenadas $(x, y)$ donde $x, y \in \{0, 1, 2, 3, 4, 5\}$. Total de 36 estados.
*   **Espacio de Acciones $\mathcal{A}$:** Discreto, correspondiente a los movimientos cardinales tácticos: $\{\text{Norte, Sur, Este, Oeste}\}$. Movimientos contra la pared mantienen al agente en su posición actual.
*   **Función de Recompensa $R(s, a, s')$:** Diseñada para capturar dos objetivos conflictivos: velocidad para terminar el nivel vs exploración para recolectar botín (loot) evadiendo peligros.
    *   *Estado Terminal:* Ubicado en $(5,5)$. Otorga una recompensa de $+100$ por completar el nivel.
    *   *Zonas de Recompensa Positiva (Loot):* Ubicadas en $(0,5)$ y $(5,0)$. Otorgan $+20$ cada una al entrar.
    *   *Zona de Penalización (Trampa/Enemigo fuerte):* Ubicada en $(3,3)$. Otorga $-50$ al pisarla.
    *   *Costo de movimiento:* $-1$ por cada paso dado, para incentivar eficiencia de movimiento.
*   **Factor de Descuento $\gamma$:** Fijado en $0.99$. En los videojuegos, queremos que el agente se dirija al objetivo a largo plazo (el estado terminal) sin que la recompensa pierda su valor rápidamente por la distancia del mapa.

#### 2. Selección de variante Monte Carlo
*   **Variante elegida:** Utilizaremos **Every-Visit Monte Carlo**.
*   **Justificación:** En un mapa de 6x6 (36 celdas), bajo una política aleatoria inicial, el agente caminará sin rumbo (random walk). Esto genera una frecuencia esperada de revisitas a los mismos estados inmensamente alta dentro de un mismo episodio antes de lograr tropezar con el estado terminal en $(5,5)$. Si usáramos *First-Visit*, estaríamos descartando el 90% de la experiencia útil recolectada en cada episodio. *Every-Visit* aprovecha cada una de las revisitas para actualizar la tabla, lo cual es mucho más eficiente en términos de datos para entornos pequeños y altamente recurrentes.

#### 3. Estrategia de Exploración
*   **Estrategia elegida:** Utilizaremos política **$\epsilon$-soft**.
*   **Justificación:** La alternativa de *Exploring Starts* asume que podemos teletransportar al agente a cualquier celda del mapa al inicio de cada episodio con probabilidad mayor a cero. En un motor de videojuegos real, esto a menudo no es razonable ni implementable sin romper la lógica del juego (los personajes tienen *spawn points* fijos). Con $\epsilon$-soft garantizamos la exploración continua sin importar dónde nazca el personaje.
*   **Valor propuesto:** Proponemos $\epsilon = 0.1$. Es lo suficientemente alto para garantizar que el agente explore los bordes y descubra las zonas de botín, pero lo suficientemente bajo (90% explotación) para que el comportamiento táctico del enemigo no se vea ridículo o puramente errático en el juego final.

#### 4. Hipótesis de comparación (MC vs Value Iteration)
*   **Hipótesis 1 (Eficiencia de Convergencia):** *Monte Carlo Control requerirá un número significativamente mayor de episodios completos para converger a la política óptima en comparación con el total de iteraciones de barrido requeridas por Value Iteration, debido a la alta varianza de las actualizaciones al final del episodio.*
*   **Hipótesis 2 (Comportamiento Seguro):** *La política óptima encontrada por Monte Carlo (con $\epsilon = 0.1$) mantendrá un radio de distancia mayor (será más conservadora) respecto a la zona de penalización de -50 que la política determinista pura hallada por Value Iteration, ya que MC debe compensar el 10% de probabilidad de ejecutar acciones aleatorias suicidas.*

---
### Task 2: Preguntas Teóricas

#### 1. Cota superior de episodios para visitar todos los pares $(s, a)$
Bajo una política uniforme aleatoria, el proceso de visitar todos los pares estado-acción se asemeja al Problema del Coleccionista de Cupones (Coupon Collector's Problem) sobre un grafo estocástico. Si tenemos $N = |\mathcal{S}| \times |\mathcal{A}|$ pares, el tiempo esperado (en pasos) para recolectar todos los "cupones" es aproximadamente $E[\text{pasos}] \approx N \ln(N)$.
Dado que las visitas ocurren a lo largo de episodios de longitud $L$, la cota superior del número de episodios esperados está dada por la función:
$$ E[\text{Episodios}] \approx \frac{|\mathcal{S}| |\mathcal{A}| \ln(|\mathcal{S}| |\mathcal{A}|)}{L_{promedio}} $$
*Evaluación numérica:* En nuestro MDP, $|\mathcal{S}| = 36$ y $|\mathcal{A}| = 4$. 
Total de pares = $144$.
Pasos esperados $\approx 144 \ln(144) \approx 144 \times 4.97 \approx 715$ pasos.
Si un episodio aleatorio dura en promedio unos $L = 70$ pasos antes de chocar con el estado terminal, necesitaríamos alrededor de $715 / 70 \approx 11$ episodios (como cota teórica en un entorno ergódico) para al menos tocar cada par una vez en esperanza.

#### 2. Varianza del retorno $G_t$
*   **Origen de la varianza:** El retorno $G_t = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{T-t-1} R_T$ es la suma de múltiples variables aleatorias. La varianza proviene de la acumulación de estocasticidad en cada paso: la aleatoriedad de la política estocástica $\pi(a|s)$ y la aleatoriedad del entorno en la función de transición $P(s'|s,a)$ y recompensa.
*   **Crecimiento con la longitud:** Dado que la varianza de una suma de variables aleatorias independientes (o semi-independientes) crece con el número de términos, mientras más largo sea el episodio (mayor sea $T-t$), más ramas de posibilidades divergen. La incertidumbre se acumula a lo largo de la trayectoria.
*   **Consecuencia en la convergencia:** Al tener una alta varianza, las muestras individuales de $G_t$ fluctúan salvajemente. Por la Ley de los Grandes Números, para que el promedio muestral de Monte Carlo converja al verdadero valor esperado $V^\pi(s)$ con un error pequeño, se requiere recolectar un tamaño de muestra (número de episodios) masivamente mayor, lo que hace que la convergencia práctica sea extremadamente lenta en episodios largos.

#### 3. Monte Carlo en tareas continuas<span style="display:none;">madagascar</span>
*   **Problema fundamental:** Monte Carlo, por definición, requiere esperar hasta el final del episodio para poder observar y calcular el retorno total $G_t$ y luego propagarlo hacia atrás. En una tarea continua (no episódica), el tiempo final $T = \infty$. El episodio literalmente nunca termina, por lo que el algoritmo se quedaría esperando infinitamente y nunca realizaría una sola actualización de valores.
*   **Modificación propuesta:** Truncamiento de horizonte (Horizon Truncation) o Rollouts Parciales. En lugar de esperar a $T=\infty$, se define un horizonte fijo artificial $H$ (ej. 100 pasos). El episodio se corta artificialmente al llegar a $H$ y se usa la suma parcial de esos 100 pasos como si fuera el $G_t$ final.
*   **Implicaciones sobre las garantías:** Al truncar la ejecución, el algoritmo pierde su garantía matemática original. El retorno muestreado se convierte en un estimador **sesgado** (biased) porque ignora por completo todas las recompensas que existen más allá del paso $H$. Monte Carlo ya no garantiza converger a la política óptima global real, sino a una aproximación sesgada por el horizonte artificial establecido.